# Switch table: the numbers

Numbers only. Figures live in `../plots/06_switch_table.ipynb`.

MPP ships three switch tables in the 1.5C smelter folder, all committed by their own developer
in one commit in September 2022. The filename the model reads carries 92 of the 132 pairs, and
the 40 it omits are all and only routes into captive power with capture. This notebook shows
what the omission costs and what the full table recovers.

In [1]:
import sys
import importlib
sys.path.append("../..")

import common
importlib.reload(common)   # pick up edits to common.py without restarting the kernel

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from common import (SCENARIOS, LABELS, COLOURS, PLANTS, ALUMINA_PER_ALUMINIUM,
                    style, emissions, production, production_by_technology, budget,
                    intensity_split, process_emission_factors, anode, power_source,
                    overlaid, panels, year_axis, save)

style()
pd.set_option("display.width", 250)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## The three tables MPP ships

In [2]:
UPSTREAM = Path("../../../mpp-upstream-reference/aluminium/data/lc/def/intermediate")
FILES = {"technology_transitions.csv": "live, the file the model reads",
         "technology_transitions_noGridtoCCS.csv": "grid to captive capture removed",
         "technology_transitions_original.csv": "the full table"}

def pairs(path):
    d = pd.read_csv(path, low_memory=False)
    return set(zip(d.technology_origin, d.technology_destination))

sets = {f: pairs(UPSTREAM / f) for f in FILES}
summary = pd.DataFrame({"switch pairs": {f: len(v) for f, v in sets.items()},
                        "what it is": FILES})
display(summary)

live, full = sets["technology_transitions.csv"], sets["technology_transitions_original.csv"]
missing = sorted(full - live)
print(f"pairs in the full table but not the live one: {len(missing)}")
print(f"of which the destination contains CCS: {sum('CCS' in b for _, b in missing)}")
print(f"pairs in the live table but not the full one: {len(live - full)}  (so full is a strict superset)")

,switch pairs,what it is
technology_transitions.csv,92,"live, the file the model reads"
technology_transitions_noGridtoCCS.csv,120,grid to captive capture removed
technology_transitions_original.csv,132,the full table


pairs in the full table but not the live one: 40
of which the destination contains CCS: 40
pairs in the live table but not the full one: 0  (so full is a strict superset)


The forty missing pairs, all of them routes into captive power with capture:

In [3]:
pd.DataFrame(missing, columns=["origin", "destination"])

,origin,destination
0,Carbon Anode + Coal,Carbon Anode + Coal+CCS
1,Carbon Anode + Coal,Carbon Anode + Natural Gas+CCS
2,Carbon Anode + Coal,Carbon Anode+CCS + Coal+CCS
3,Carbon Anode + Coal,Carbon Anode+CCS + Natural Gas+CCS
4,Carbon Anode + Coal,Inert Anode + Coal+CCS
5,Carbon Anode + Coal,Inert Anode + Natural Gas+CCS
6,Carbon Anode + Coal+CCS,Carbon Anode + Coal+CCS
7,Carbon Anode + Coal+CCS,Carbon Anode+CCS + Coal+CCS
8,Carbon Anode + Coal+CCS,Inert Anode + Coal+CCS
9,Carbon Anode + Grid,Carbon Anode + Coal+CCS


## Grid reverting to captive fossil

All three tables carry the same twelve routes by which a grid-connected plant reverts to
unabated captive fossil generation. `noGridtoCCS` removes only the capture versions, so it
closes the better move and keeps the worse one. Choice 33 records the decision to leave both
open.

In [4]:
GRID = {"Grid", "PPA+Grid"}
UNABATED = {"Coal", "Natural Gas"}
ABATED = {"Coal+CCS", "Natural Gas+CCS"}

rows = []
for f, ps in sets.items():
    real = [(a, b) for a, b in ps if b != "decommission" and a != "New-build"]
    rows.append({"file": f,
                 "grid to captive, no capture": sum(power_source(a) in GRID and power_source(b) in UNABATED for a, b in real),
                 "grid to captive, with capture": sum(power_source(a) in GRID and power_source(b) in ABATED for a, b in real)})
pd.DataFrame(rows).set_index("file")

,"grid to captive, no capture","grid to captive, with capture"
file,,
technology_transitions.csv,12,0
technology_transitions_noGridtoCCS.csv,12,0
technology_transitions_original.csv,12,12


## Whether those routes actually fire

In [5]:
def reversals(scenario):
    """Plants moving from a grid power source to captive fossil, by capture status."""
    prev, found = None, []
    for year in range(2020, 2051):
        path = Path("../..") / "runs" / scenario / "smelter" / "stack_tracker" / f"stack_{year}.csv"
        stack = pd.read_csv(path).set_index("uuid")
        if prev is not None:
            common = prev.index.intersection(stack.index)
            moved = common[prev.loc[common, "technology"].values != stack.loc[common, "technology"].values]
            for uuid in moved:
                origin, dest = prev.loc[uuid, "technology"], stack.loc[uuid, "technology"]
                if power_source(origin) in GRID and power_source(dest) in (UNABATED | ABATED):
                    found.append({"year": year, "origin": origin, "destination": dest,
                                  "capacity": stack.loc[uuid, "annual_production_capacity"],
                                  "to_capture": power_source(dest) in ABATED})
        prev = stack
    return pd.DataFrame(found)

rows = []
for s in SCENARIOS:
    r = reversals(s)
    rows.append({"Scenario": LABELS[s], "plants": len(r),
                 "to capture, Mt": r[r.to_capture].capacity.sum() if len(r) else 0.0,
                 "uncaptured, Mt": r[~r.to_capture].capacity.sum() if len(r) else 0.0})
print("On the full table the uncaptured reversals stop firing entirely.")
pd.DataFrame(rows).set_index("Scenario").round(2)

On the full table the uncaptured reversals stop firing entirely.


,plants,"to capture, Mt","uncaptured, Mt"
Scenario,,,
Business as usual,28,1.36,7.67
"MPP grid, inert anode locked",11,2.28,0.00
"SBTi grid, inert anode locked",0,0.00,0.00
"MPP grid, inert anode unlocked",6,0.97,0.00
"SBTi grid, inert anode unlocked",0,0.00,0.00
